# RAASTA · Flooded-road intelligence
### A Google Colab deep learning research prototype
**Author:** Allah Ditta · Computer Science  
**Workflow:** public dataset → aligned image pairs → training → independent evaluation → prediction → inspection-planning experiment.

This notebook trains a six-channel U-Net on **SpaceNet 8 before/after satellite images** to segment background, annotated roads without a flood label, and flooded roads. It includes a prediction viewer, GeoTIFF export, a geographic map, and a **separate synthetic road-network experiment driven by model scores**.

**Scope:** historical Germany/Louisiana imagery. Pakistan performance is unverified. “Not labelled flooded” never means safe to drive. The map is dated research evidence, not live navigation. The planning experiment has fictional depots, communities, costs, and roads; it is not a mapped real delivery network.

## Start here
1. Upload this `.ipynb` at [Google Colab](https://colab.research.google.com/).
2. Select **Runtime → Change runtime type → GPU** if available.
3. Run cells in order. In configuration, leave `PROFILE = "starter"` for an initial experiment. `smoke` only checks the pipeline; `full` uses more data.
4. Optional Drive mounting preserves checkpoints. Downloaded files and processed images are saved in Drive when `USE_DRIVE=True`.
5. First read the dataset size estimate printed before download. This is a substantial imagery dataset; start small.

**No accuracy is pre-filled.** Training/evaluation outputs are created by your run. A small starter run establishes functionality, not production quality.


## 1 · Install lightweight dependencies
Colab normally includes PyTorch. We keep its installed build rather than replacing CUDA libraries. Rasterio handles alignment and georeferencing. No GitHub account, Kaggle token, or AWS key is required.


In [ ]:
import sys, subprocess, importlib.util
packages = {"rasterio":"rasterio>=1.3,<2", "shapely":"shapely>=2,<3", "pyproj":"pyproj>=3,<4",
            "pandas":"pandas>=2,<3", "matplotlib":"matplotlib>=3.8,<4", "networkx":"networkx>=3,<4",
            "requests":"requests>=2.31,<3", "folium":"folium>=0.16,<1", "ipywidgets":"ipywidgets>=8,<9"}
missing = [spec for module,spec in packages.items() if importlib.util.find_spec(module) is None]
if missing: subprocess.check_call([sys.executable,"-m","pip","install","-q",*missing])
if importlib.util.find_spec("torch") is None:
    raise RuntimeError("Select a standard Colab Python runtime with PyTorch, then rerun this cell.")


##  · Configuration and persistent outputs
Starter: 60 training, 20 validation, 24 test tiles, 20 epochs. The full profile raises these caps and training duration. Download sizes are measured from the public bucket before transfer. With `USE_DRIVE=True`, raw imagery is kept in `MyDrive/Raasta/dataset/raw`, processed arrays in `MyDrive/Raasta/dataset/cache_v1`, and checkpoints/results in `MyDrive/Raasta/runs/<RUN_NAME>`. Make sure Drive has enough free space for raw and processed data.

**Splits:** train/validation use separated east-west tile columns in Germany, with a held-out boundary column gap. Louisiana-East is an independent geographic test set. Selection is deterministic and never uses test performance. This is a small cross-region benchmark, not evidence about Pakistan.

Set `USE_DRIVE = False` if you do not want to mount Drive; download the result archive before ending your session. To begin a new independent run, change `RUN_NAME` or set `RESUME = False` and use a new output directory.


In [ ]:
from pathlib import Path
import sys, os, json, csv, io, random, time, hashlib, math, warnings, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject, transform_geom, transform_bounds
from rasterio.features import rasterize
from rasterio.transform import Affine
from shapely.geometry import shape, mapping
from shapely.ops import transform as shape_transform
from pyproj import CRS, Transformer
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from concurrent.futures import ThreadPoolExecutor
from IPython.display import display

PROFILE = "starter" # "smoke", "starter", or "full"
USE_DRIVE = True
RESUME = True
SEED = 42
TILE_SIZE = 512
PATCH_SIZE = 256
ROAD_BUFFER_METRES = 3.0 # radius: 6 m proxy corridor, not measured road width
BATCH_SIZE = 4
MAX_DOWNLOAD_GB = 6.0 # hard guard; raise deliberately for the full profile
RUN_NAME = "raasta_v1_" + PROFILE
settings = {"smoke": (4,2,2,2), "starter": (60,20,24,20), "full": (150,40,80,40)}
N_TRAIN,N_VAL,N_TEST,EPOCHS = settings[PROFILE]
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = False
WORK = Path("/content/raasta")
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive/Raasta")
    OUT = PROJECT / "runs" / RUN_NAME
else:
    PROJECT = WORK
    OUT = PROJECT / "runs" / RUN_NAME
RAW, CACHE = PROJECT/"dataset"/"raw", PROJECT/"dataset"/"cache_v1"
for folder in [RAW,CACHE,OUT]: folder.mkdir(parents=True,exist_ok=True)
# Preserve checkpoints created by the earlier notebook layout.
legacy = PROJECT / RUN_NAME
if USE_DRIVE and legacy.is_dir() and not (OUT/"last_checkpoint.pt").exists():
    for old_file in legacy.iterdir():
        if old_file.is_file() and not (OUT/old_file.name).exists():
            shutil.copy2(old_file,OUT/old_file.name)
    print("Copied earlier run outputs into the persistent run folder.")
print("Device:",DEVICE,"| outputs:",OUT)
if DEVICE.type == "cpu": print("CPU runtime: use smoke for verification, then switch to a GPU for training.")
versions = {"python":sys.version,"torch":torch.__version__,"numpy":np.__version__,"rasterio":rasterio.__version__}
(OUT/"environment.json").write_text(json.dumps(versions,indent=2))

print("Persistent raw dataset:", RAW)
print("Persistent processed dataset:", CACHE)
print("Keep USE_DRIVE=True and RESUME=True to preserve and continue your work.")


## Continue a previous session — next-day shortcut
**First day:** leave `CONTINUE_PREVIOUS=False` and run all cells in order.

**Next day:** run **Install dependencies**, **Configuration and persistent outputs** (mount the same Drive account), then this cell with `CONTINUE_PREVIOUS=True`. Keep your original `PROFILE`, `RUN_NAME`, image sizes and batch size. This reconstructs Python functions, reloads the cached dataset and the saved model without downloading imagery again.

Then jump directly to **Train** to continue training, **Independent evaluation** to calculate results, or **Prediction** to demonstrate the saved best model. The widgets and map must be recreated when you need them. For the synthetic experiment, first select a prediction tile. If you only completed part of the first-day setup, rerun the relevant preparation cells; existing complete downloads are reused.

Your notebook file preserves code; this cell rebuilds the runtime state. It does not keep the old Colab runtime alive.


In [ ]:
RESTORE_DEFINITIONS = 'import xml.etree.ElementTree as ET\nfrom urllib.parse import quote\nBASE = "https://spacenet-dataset.s3.amazonaws.com/"\nPREFIX = "spacenet/SN8_floods/"\nAREAS = ["Germany_Training_Public", "Louisiana-East_Training_Public"]\n\ndef session():\n    s=requests.Session()\n    retry=Retry(total=4,backoff_factor=1,status_forcelist=[429,500,502,503,504])\n    s.mount("https://",HTTPAdapter(max_retries=retry))\n    return s\n\ndef list_objects(prefix):\n    rows=[]; token=None; ns={"s":"http://s3.amazonaws.com/doc/2006-03-01/"}\n    with session() as s:\n        while True:\n            params={"list-type":"2","prefix":prefix,"max-keys":1000}\n            if token: params["continuation-token"]=token\n            r=s.get(BASE,params=params,timeout=(15,90)); r.raise_for_status()\n            root=ET.fromstring(r.content)\n            rows.extend({"key":v.findtext("s:Key",namespaces=ns),"size":int(v.findtext("s:Size",namespaces=ns))}\n                        for v in root.findall("s:Contents",ns))\n            token=root.findtext("s:NextContinuationToken",namespaces=ns)\n            if not token: break\n    return rows\n\ndef fetch(key, expected=None):\n    relative=Path(key).relative_to(PREFIX)\n    p=RAW/relative\n    if p.exists() and (p.stat().st_size==expected if expected is not None else p.stat().st_size>0): return p\n    p.parent.mkdir(parents=True,exist_ok=True)\n    tmp=p.with_suffix(p.suffix+".part")\n    with session() as s, s.get(BASE+quote(key,safe="/"),stream=True,timeout=(20,120)) as r:\n        r.raise_for_status()\n        with tmp.open("wb") as f:\n            for chunk in r.iter_content(1024*1024): f.write(chunk)\n    if expected is not None and tmp.stat().st_size != expected:\n        raise IOError("Incomplete download: "+key)\n    tmp.replace(p)\n    return p\n\n\n\nCLASS_NAMES=["Background","Road: no flood label","Flooded road"]\nPALETTE=np.array([[22,32,46],[41,182,164],[250,92,78]],dtype=np.uint8)\nIGNORE=255\n\ndef local_path(key): return RAW/Path(key).relative_to(PREFIX)\n\ndef load_pair(pre_path,post_path,size=TILE_SIZE):\n    with rasterio.open(pre_path) as src:\n        if src.crs is None: raise ValueError("Pre-event image has no CRS.")\n        if src.count<3 or src.dtypes[0]!="uint8": raise ValueError("Expected 8-bit RGB SpaceNet imagery.")\n        crs=src.crs; trans=src.transform*Affine.scale(src.width/size,src.height/size)\n        pre=src.read([1,2,3],out_shape=(3,size,size),resampling=Resampling.bilinear)\n        valid_pre=src.dataset_mask(out_shape=(size,size),resampling=Resampling.nearest)>0\n    post=np.zeros_like(pre); valid_post=np.zeros((size,size),np.uint8)\n    with rasterio.open(post_path) as src:\n        if src.crs is None or src.count<3 or src.dtypes[0]!="uint8": raise ValueError("Expected georeferenced 8-bit RGB post image.")\n        for c in range(3):\n            reproject(source=rasterio.band(src,c+1),destination=post[c],src_transform=src.transform,src_crs=src.crs,\n                      dst_transform=trans,dst_crs=crs,resampling=Resampling.bilinear)\n        reproject(source=src.dataset_mask(),destination=valid_post,src_transform=src.transform,src_crs=src.crs,\n                  dst_transform=trans,dst_crs=crs,resampling=Resampling.nearest)\n    valid=valid_pre & (valid_post>0) & (pre.max(0)>0) & (post.max(0)>0)\n    if valid.mean()<0.1: raise ValueError("Less than 10% paired coverage. Check image matching and CRS.")\n    return np.concatenate([pre,post]),valid,trans,crs\n\ndef road_geometries(label_path,crs):\n    data=json.loads(Path(label_path).read_text())\n    roads=[f for f in data["features"] if f.get("geometry") and f.get("properties",{}).get("highway")\n           and f["geometry"]["type"] in ("LineString","MultiLineString")]\n    if not roads:return []\n    declared=str(data.get("crs",{}))\n    if declared and not any(s in declared for s in ["CRS84","4326"]):\n        raise ValueError("Unexpected annotation CRS; do not assume longitude/latitude.")\n    centre=shape(roads[0]["geometry"]).centroid\n    zone=max(1,min(60,int((centre.x+180)//6)+1)); epsg=(32600 if centre.y>=0 else 32700)+zone\n    to_metric=Transformer.from_crs(4326,epsg,always_xy=True).transform\n    to_grid=Transformer.from_crs(epsg,crs,always_xy=True).transform\n    result=[]\n    for f in roads:\n        flag=str(f["properties"].get("flooded")).strip().lower()\n        if flag not in {"yes","1","true","no","0","false","none","","null"}: raise ValueError("Unknown flood value: "+flag)\n        cls=2 if flag in {"yes","1","true"} else 1\n        poly=shape_transform(to_grid,shape_transform(to_metric,shape(f["geometry"])).buffer(ROAD_BUFFER_METRES))\n        result.append((mapping(poly),cls))\n    return sorted(result,key=lambda pair:pair[1])\n\ndef prepare_row(row):\n    identity=f"{row.area}/{row.tile}/{row.post_key}/{TILE_SIZE}/{ROAD_BUFFER_METRES}/v1"\n    p=CACHE/(hashlib.sha256(identity.encode()).hexdigest()[:20]+".npz")\n    if not p.exists():\n        x,valid,trans,crs=load_pair(local_path(row.pre_key),local_path(row.post_key))\n        geoms=road_geometries(local_path(row.label_key),crs)\n        mask=rasterize(geoms,out_shape=(TILE_SIZE,TILE_SIZE),transform=trans,fill=0,dtype="uint8") if geoms else np.zeros((TILE_SIZE,TILE_SIZE),np.uint8)\n        mask[~valid]=IGNORE\n        tmp=p.with_suffix(".part.npz")\n        np.savez_compressed(tmp,x=x,y=mask,transform=np.array(tuple(trans)[:6]),crs=np.array(crs.to_string()))\n        tmp.replace(p)\n    return str(p)\n\n\nclass RoadDataset(Dataset):\n    def __init__(self,frame,training=False): self.rows=list(frame.itertuples());self.training=training\n    def __len__(self): return len(self.rows)*(4 if self.training else 1)\n    def __getitem__(self,i):\n        with np.load(self.rows[i%len(self.rows)].cache) as z:x=z["x"].copy();y=z["y"].copy()\n        if self.training:\n            h,w=y.shape; ys,xs=np.where((y==1)|(y==2))\n            if len(ys) and random.random()<.5:\n                j=random.randrange(len(ys));top=int(np.clip(ys[j]-PATCH_SIZE//2,0,h-PATCH_SIZE));left=int(np.clip(xs[j]-PATCH_SIZE//2,0,w-PATCH_SIZE))\n            else:top=random.randint(0,h-PATCH_SIZE);left=random.randint(0,w-PATCH_SIZE)\n            x=x[:,top:top+PATCH_SIZE,left:left+PATCH_SIZE];y=y[top:top+PATCH_SIZE,left:left+PATCH_SIZE]\n            k=random.randrange(4);x=np.rot90(x,k,axes=(1,2));y=np.rot90(y,k)\n            if random.random()<.5:x=x[:,:,::-1];y=y[:,::-1]\n        return torch.from_numpy(np.ascontiguousarray(x)).float()/255.,torch.from_numpy(np.ascontiguousarray(y)).long()\n\nloaders={s:DataLoader(RoadDataset(manifest[manifest.split==s],s=="train"),\n             batch_size=BATCH_SIZE if s=="train" else 1,shuffle=s=="train",num_workers=0,\n             pin_memory=DEVICE.type=="cuda") for s in ["train","val","test"]}\nclass Block(nn.Module):\n    def __init__(self,a,b):\n        super().__init__();self.net=nn.Sequential(nn.Conv2d(a,b,3,padding=1),nn.GroupNorm(4,b),nn.ReLU(),\n                                                 nn.Conv2d(b,b,3,padding=1),nn.GroupNorm(4,b),nn.ReLU())\n    def forward(self,x):return self.net(x)\nclass RoadUNet(nn.Module):\n    def __init__(self):\n        super().__init__();self.e1=Block(6,16);self.e2=Block(16,32);self.e3=Block(32,64);self.mid=Block(64,128)\n        self.d3=Block(192,64);self.d2=Block(96,32);self.d1=Block(48,16);self.head=nn.Conv2d(16,3,1)\n    def forward(self,x):\n        a=self.e1(x);b=self.e2(nn.functional.max_pool2d(a,2));c=self.e3(nn.functional.max_pool2d(b,2))\n        z=self.mid(nn.functional.max_pool2d(c,2))\n        for skip,block in [(c,self.d3),(b,self.d2),(a,self.d1)]:\n            z=nn.functional.interpolate(z,size=skip.shape[-2:],mode="bilinear",align_corners=False)\n            z=block(torch.cat([z,skip],dim=1))\n        return self.head(z)\nmodel=RoadUNet().to(DEVICE)\nfreq=counts["train"]/counts["train"].sum()\nweights=np.clip(1/np.sqrt(freq),1,30);weights=weights/weights.mean()\nWEIGHTS=torch.tensor(weights,dtype=torch.float32,device=DEVICE)\ndef loss_fn(logits,y):\n    valid=y!=IGNORE\n    if not valid.any():return logits.sum()*0\n    ce=nn.functional.cross_entropy(logits,y,weight=WEIGHTS,ignore_index=IGNORE)\n    p=logits.softmax(1); terms=[]\n    for c in [1,2]:\n        target=(y==c).float(); pc=p[:,c]*valid\n        terms.append(1-(2*(pc*target).sum()+1)/(pc.sum()+target.sum()+1))\n    return ce+.5*torch.stack(terms).mean()\nprint(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}")\n\n\ndef confusion_update(cm,pred,y):\n    valid=y!=IGNORE\n    return cm+torch.bincount(3*y[valid]+pred[valid],minlength=9).reshape(3,3).cpu().numpy()\ndef metrics(cm):\n    tp=np.diag(cm).astype(float);actual=cm.sum(1);predicted=cm.sum(0);union=actual+predicted-tp\n    div=lambda a,b:np.divide(a,b,out=np.full_like(a,np.nan,dtype=float),where=b>0)\n    return pd.DataFrame({"class":CLASS_NAMES,"IoU":div(tp,union),"precision":div(tp,predicted),\n                         "recall":div(tp,actual),"Dice_F1":div(2*tp,actual+predicted),"support_pixels":actual})\n@torch.no_grad()\ndef evaluate(loader):\n    model.eval();cm=np.zeros((3,3),np.int64);total=0.;n=0\n    for x,y in loader:\n        x=x.to(DEVICE);y=y.to(DEVICE);logits=model(x)\n        total+=loss_fn(logits,y).item();n+=1\n        cm=confusion_update(cm,logits.argmax(1),y)\n    return total/max(n,1),cm\n'
CONTINUE_PREVIOUS = False  # Set True on your next-day session; keep False on the first run.
if CONTINUE_PREVIOUS:
    manifest_path=OUT/"manifest.csv"
    config_path=OUT/"config.json"
    if not manifest_path.exists() or not config_path.exists():
        raise RuntimeError("No complete previous setup found. Run discovery, preparation and training cells first.")
    saved_config=json.loads(config_path.read_text())
    expected={"seed":SEED,"tile_size":TILE_SIZE,"patch_size":PATCH_SIZE,"road_buffer_metres":ROAD_BUFFER_METRES,
              "profile":PROFILE,"batch_size":BATCH_SIZE,"model":"six_channel_unet_v1"}
    for k,v in expected.items():
        if saved_config.get(k)!=v:
            raise ValueError(f"Saved setting {k} differs. Restore your original settings or choose a new RUN_NAME.")
    manifest=pd.read_csv(manifest_path)
    # Rebuild paths so the same run also works after reconnecting to Drive.
    manifest["cache"]=[str(CACHE/Path(v).name) for v in manifest["cache"]]
    missing=[v for v in manifest.cache if not Path(v).exists()]
    if missing:raise RuntimeError("Some prepared files are missing. Rerun discovery and preprocessing to rebuild them.")
    counts={s:np.zeros(3,np.int64) for s in ["train","val","test"]}
    for r in manifest.itertuples():
        with np.load(r.cache) as z:
            y=z["y"];counts[r.split]+=np.bincount(y[y!=255],minlength=3)
    exec(RESTORE_DEFINITIONS,globals())
    config=saved_config
    signature=hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()
    best_path=OUT/"best_model.pt";last=OUT/"last_checkpoint.pt"
    history=[];best=-1.
    if last.exists():
        ck=torch.load(last,map_location="cpu",weights_only=False)
        if ck["signature"]!=signature:raise ValueError("Checkpoint and saved configuration disagree.")
        history=ck["history"];best=ck["best"]
        print(f"Last completed epoch: {ck['epoch']+1}. The training cell resumes the optimizer and RNG states.")
    if best_path.exists():
        ck_best=torch.load(best_path,map_location=DEVICE,weights_only=False)
        if ck_best["signature"]!=signature:raise ValueError("Best model belongs to a different run.")
        model.load_state_dict(ck_best["model"]);model.eval()
        print("Best model loaded for evaluation and predictions.")
    elif last.exists():
        model.load_state_dict(ck["model"])
    else:print("Prepared data restored; run the training cell to create your first checkpoint.")
    metrics_path=OUT/"test_metrics.csv"
    if metrics_path.exists():results=pd.read_csv(metrics_path)
    print("Session restored. Skip download, preparation and model-definition cells.")
    print("Continue at TRAINING, or EVALUATION, or PREDICTION. No dataset re-download is required.")
else:
    print("First run: continue to dataset discovery below. Next day, set CONTINUE_PREVIOUS=True in this cell.")


## 3 · Discover the official files and create a reproducible manifest
Only selected image pairs and their labels are downloaded. Existing complete downloads are reused; incomplete files are retried. We use the official pairing CSV, not filename guesses. If the publisher changes its format, the notebook stops with a clear error.


In [ ]:
import xml.etree.ElementTree as ET
from urllib.parse import quote
BASE = "https://spacenet-dataset.s3.amazonaws.com/"
PREFIX = "spacenet/SN8_floods/"
AREAS = ["Germany_Training_Public", "Louisiana-East_Training_Public"]

def session():
    s=requests.Session()
    retry=Retry(total=4,backoff_factor=1,status_forcelist=[429,500,502,503,504])
    s.mount("https://",HTTPAdapter(max_retries=retry))
    return s

def list_objects(prefix):
    rows=[]; token=None; ns={"s":"http://s3.amazonaws.com/doc/2006-03-01/"}
    with session() as s:
        while True:
            params={"list-type":"2","prefix":prefix,"max-keys":1000}
            if token: params["continuation-token"]=token
            r=s.get(BASE,params=params,timeout=(15,90)); r.raise_for_status()
            root=ET.fromstring(r.content)
            rows.extend({"key":v.findtext("s:Key",namespaces=ns),"size":int(v.findtext("s:Size",namespaces=ns))}
                        for v in root.findall("s:Contents",ns))
            token=root.findtext("s:NextContinuationToken",namespaces=ns)
            if not token: break
    return rows

def fetch(key, expected=None):
    relative=Path(key).relative_to(PREFIX)
    p=RAW/relative
    if p.exists() and (p.stat().st_size==expected if expected is not None else p.stat().st_size>0): return p
    p.parent.mkdir(parents=True,exist_ok=True)
    tmp=p.with_suffix(p.suffix+".part")
    with session() as s, s.get(BASE+quote(key,safe="/"),stream=True,timeout=(20,120)) as r:
        r.raise_for_status()
        with tmp.open("wb") as f:
            for chunk in r.iter_content(1024*1024): f.write(chunk)
    if expected is not None and tmp.stat().st_size != expected:
        raise IOError("Incomplete download: "+key)
    tmp.replace(p)
    return p

indexes={a:list_objects(PREFIX+a+"/") for a in AREAS}
SIZES={r["key"]:r["size"] for rows in indexes.values() for r in rows}

def area_table(area):
    keys=[r["key"] for r in indexes[area]]
    lookup={Path(k).name:k for k in keys}
    mapping_key=next(k for k in keys if k.endswith("label_image_mapping.csv"))
    table=pd.read_csv(fetch(mapping_key,SIZES[mapping_key])).fillna("")
    required=["label","pre-event image","post-event image 1"]
    if not set(required).issubset(table.columns): raise ValueError(f"Unexpected mapping columns: {table.columns}")
    rows=[]
    for _,r in table.iterrows():
        names=[Path(str(r[c])).name for c in required]
        if not all(n in lookup for n in names): continue
        tile=Path(names[0]).stem
        parts=tile.split("_")
        rows.append({"area":area,"tile":tile,"column":int(parts[-2]),
                     "label_key":lookup[names[0]],"pre_key":lookup[names[1]],"post_key":lookup[names[2]]})
    return pd.DataFrame(rows).sort_values("tile").reset_index(drop=True)

germany, louisiana = [area_table(a) for a in AREAS]
columns=sorted(germany.column.unique())
cut=columns[int(0.75*len(columns))]
train_pool=germany[germany.column < cut-1]
val_pool=germany[germany.column > cut]
def choose(df,n,split):
    if len(df)<n: print(f"{split}: capped to {len(df)} available tiles.")
    x=df.sample(n=min(n,len(df)),random_state=SEED).copy(); x["split"]=split
    return x
manifest=pd.concat([choose(train_pool,N_TRAIN,"train"),choose(val_pool,N_VAL,"val"),
                    choose(louisiana,N_TEST,"test")],ignore_index=True)
assert manifest.groupby("split").size().min()>0
assert not set(manifest.query('split=="train"').pre_key) & set(manifest.query('split=="val"').pre_key)
keys=sorted(set(manifest[["pre_key","post_key","label_key"]].values.ravel()))
size_gb=sum(SIZES[k] for k in keys)/1e9
print(manifest.groupby(["split","area"]).size())
print(f"Selected raw files: {size_gb:.2f} GB. Germany excluded boundary columns: {cut-1} and {cut}.")
if size_gb>MAX_DOWNLOAD_GB:
    raise RuntimeError(f"Download needs {size_gb:.2f} GB; lower tile counts or increase MAX_DOWNLOAD_GB and rerun configuration onward.")
if shutil.disk_usage(WORK).free < sum(SIZES[k] for k in keys)+2_000_000_000:
    raise RuntimeError("Not enough temporary disk space for this profile.")
manifest.to_csv(OUT/"manifest.csv",index=False)
provenance={"dataset":"SpaceNet 8","official_page":"https://www.spacenet.ai/sn8-challenge/",
            "source_bucket":BASE,"source_prefix":PREFIX,"selected_raw_bytes":sum(SIZES[k] for k in keys),
            "downloaded_files":[{"key":k,"bytes":SIZES[k]} for k in keys],
            "storage_raw":str(RAW),"storage_processed":str(CACHE),"regions":AREAS,
            "note":"Historical Germany/Louisiana imagery; Pakistan validation not performed."}
(OUT/"dataset_source.json").write_text(json.dumps(provenance,indent=2))
with ThreadPoolExecutor(max_workers=4) as pool:
    for i,p in enumerate(pool.map(lambda k:fetch(k,SIZES[k]),keys),1):
        if i%30==0 or i==len(keys): print(f"Downloaded/verified {i}/{len(keys)} files")


## 4 · Align images and build road masks
Both images are warped onto the **same pre-event reference grid**. This is essential: equal image dimensions do not guarantee geographic alignment.

Road annotations are centre lines. We buffer each line by 3 metres on each side in a local UTM projection, then rasterize the corridor. This fixed-width label proxy is an explicit modelling choice. `flooded=yes/1/true` is positive; absent flood attributes follow the official baseline convention as non-flood-labelled roads. This does not establish passability. Unknown values cause a failure rather than silently changing labels.

Classes: **0 background, 1 road without flood label, 2 flooded road, 255 no valid paired imagery**. Road masks take precedence over background. Flood labels take precedence at overlapping corridors. RGB values are scaled from the documented 8-bit imagery; unexpected image types fail for inspection.


In [ ]:
CLASS_NAMES=["Background","Road: no flood label","Flooded road"]
PALETTE=np.array([[22,32,46],[41,182,164],[250,92,78]],dtype=np.uint8)
IGNORE=255

def local_path(key): return RAW/Path(key).relative_to(PREFIX)

def load_pair(pre_path,post_path,size=TILE_SIZE):
    with rasterio.open(pre_path) as src:
        if src.crs is None: raise ValueError("Pre-event image has no CRS.")
        if src.count<3 or src.dtypes[0]!="uint8": raise ValueError("Expected 8-bit RGB SpaceNet imagery.")
        crs=src.crs; trans=src.transform*Affine.scale(src.width/size,src.height/size)
        pre=src.read([1,2,3],out_shape=(3,size,size),resampling=Resampling.bilinear)
        valid_pre=src.dataset_mask(out_shape=(size,size),resampling=Resampling.nearest)>0
    post=np.zeros_like(pre); valid_post=np.zeros((size,size),np.uint8)
    with rasterio.open(post_path) as src:
        if src.crs is None or src.count<3 or src.dtypes[0]!="uint8": raise ValueError("Expected georeferenced 8-bit RGB post image.")
        for c in range(3):
            reproject(source=rasterio.band(src,c+1),destination=post[c],src_transform=src.transform,src_crs=src.crs,
                      dst_transform=trans,dst_crs=crs,resampling=Resampling.bilinear)
        reproject(source=src.dataset_mask(),destination=valid_post,src_transform=src.transform,src_crs=src.crs,
                  dst_transform=trans,dst_crs=crs,resampling=Resampling.nearest)
    valid=valid_pre & (valid_post>0) & (pre.max(0)>0) & (post.max(0)>0)
    if valid.mean()<0.1: raise ValueError("Less than 10% paired coverage. Check image matching and CRS.")
    return np.concatenate([pre,post]),valid,trans,crs

def road_geometries(label_path,crs):
    data=json.loads(Path(label_path).read_text())
    roads=[f for f in data["features"] if f.get("geometry") and f.get("properties",{}).get("highway")
           and f["geometry"]["type"] in ("LineString","MultiLineString")]
    if not roads:return []
    declared=str(data.get("crs",{}))
    if declared and not any(s in declared for s in ["CRS84","4326"]):
        raise ValueError("Unexpected annotation CRS; do not assume longitude/latitude.")
    centre=shape(roads[0]["geometry"]).centroid
    zone=max(1,min(60,int((centre.x+180)//6)+1)); epsg=(32600 if centre.y>=0 else 32700)+zone
    to_metric=Transformer.from_crs(4326,epsg,always_xy=True).transform
    to_grid=Transformer.from_crs(epsg,crs,always_xy=True).transform
    result=[]
    for f in roads:
        flag=str(f["properties"].get("flooded")).strip().lower()
        if flag not in {"yes","1","true","no","0","false","none","","null"}: raise ValueError("Unknown flood value: "+flag)
        cls=2 if flag in {"yes","1","true"} else 1
        poly=shape_transform(to_grid,shape_transform(to_metric,shape(f["geometry"])).buffer(ROAD_BUFFER_METRES))
        result.append((mapping(poly),cls))
    return sorted(result,key=lambda pair:pair[1])

def prepare_row(row):
    identity=f"{row.area}/{row.tile}/{row.post_key}/{TILE_SIZE}/{ROAD_BUFFER_METRES}/v1"
    p=CACHE/(hashlib.sha256(identity.encode()).hexdigest()[:20]+".npz")
    if not p.exists():
        x,valid,trans,crs=load_pair(local_path(row.pre_key),local_path(row.post_key))
        geoms=road_geometries(local_path(row.label_key),crs)
        mask=rasterize(geoms,out_shape=(TILE_SIZE,TILE_SIZE),transform=trans,fill=0,dtype="uint8") if geoms else np.zeros((TILE_SIZE,TILE_SIZE),np.uint8)
        mask[~valid]=IGNORE
        tmp=p.with_suffix(".part.npz")
        np.savez_compressed(tmp,x=x,y=mask,transform=np.array(tuple(trans)[:6]),crs=np.array(crs.to_string()))
        tmp.replace(p)
    return str(p)
manifest["cache"]=[prepare_row(r) for r in manifest.itertuples()]
manifest.to_csv(OUT/"manifest.csv",index=False)
counts={s:np.zeros(3,np.int64) for s in ["train","val","test"]}
for r in manifest.itertuples():
    with np.load(r.cache) as z: y=z["y"]; counts[r.split]+=np.bincount(y[y!=IGNORE],minlength=3)
display(pd.DataFrame(counts,index=CLASS_NAMES))
if (counts["train"]==0).any():
    raise RuntimeError("Training subset lacks a class. Increase N_TRAIN/profile, then rerun from configuration.")
for s in ["val","test"]:
    if (counts[s]==0).any(): warnings.warn(f"{s} lacks a class: affected metrics may be undefined. Increase this split.")
print("Prepared",len(manifest),"aligned pairs.")


## 5 · Inspect the actual training data
Check alignment and masks before spending time on training. Coloured corridors are training labels, not predictions.


In [ ]:
def show_tile(row):
    with np.load(row.cache) as z: x=z["x"];y=z["y"]
    fig,ax=plt.subplots(1,3,figsize=(15,5))
    ax[0].imshow(x[:3].transpose(1,2,0));ax[0].set_title("Before")
    ax[1].imshow(x[3:].transpose(1,2,0));ax[1].set_title("After")
    ax[2].imshow(x[3:].transpose(1,2,0));ax[2].imshow(np.ma.masked_where((y==0)|(y==IGNORE),y),vmin=0,vmax=2,cmap=plt.matplotlib.colors.ListedColormap(PALETTE/255),alpha=.7)
    ax[2].set_title("Labels: teal road / coral flooded road")
    for a in ax:a.axis("off")
    fig.suptitle(f"{row.area} · {row.tile} · {row.split}");plt.tight_layout();plt.show()
show_tile(manifest.query('split=="train"').iloc[0])


## 6 · Training samples and U-Net
The model receives six channels (before RGB + after RGB). It learns **three-class segmentation** from scratch. Half the training crops are centred near annotated roads to reduce empty-background dominance. Validation and test evaluate complete resized tiles with no random crops or augmentation.

The loss combines training-frequency-weighted cross entropy with road-class Dice loss. All augmentations apply identically to both images and the mask. No data from validation or test contributes to class weights.


In [ ]:
class RoadDataset(Dataset):
    def __init__(self,frame,training=False): self.rows=list(frame.itertuples());self.training=training
    def __len__(self): return len(self.rows)*(4 if self.training else 1)
    def __getitem__(self,i):
        with np.load(self.rows[i%len(self.rows)].cache) as z:x=z["x"].copy();y=z["y"].copy()
        if self.training:
            h,w=y.shape; ys,xs=np.where((y==1)|(y==2))
            if len(ys) and random.random()<.5:
                j=random.randrange(len(ys));top=int(np.clip(ys[j]-PATCH_SIZE//2,0,h-PATCH_SIZE));left=int(np.clip(xs[j]-PATCH_SIZE//2,0,w-PATCH_SIZE))
            else:top=random.randint(0,h-PATCH_SIZE);left=random.randint(0,w-PATCH_SIZE)
            x=x[:,top:top+PATCH_SIZE,left:left+PATCH_SIZE];y=y[top:top+PATCH_SIZE,left:left+PATCH_SIZE]
            k=random.randrange(4);x=np.rot90(x,k,axes=(1,2));y=np.rot90(y,k)
            if random.random()<.5:x=x[:,:,::-1];y=y[:,::-1]
        return torch.from_numpy(np.ascontiguousarray(x)).float()/255.,torch.from_numpy(np.ascontiguousarray(y)).long()

loaders={s:DataLoader(RoadDataset(manifest[manifest.split==s],s=="train"),
             batch_size=BATCH_SIZE if s=="train" else 1,shuffle=s=="train",num_workers=0,
             pin_memory=DEVICE.type=="cuda") for s in ["train","val","test"]}
class Block(nn.Module):
    def __init__(self,a,b):
        super().__init__();self.net=nn.Sequential(nn.Conv2d(a,b,3,padding=1),nn.GroupNorm(4,b),nn.ReLU(),
                                                 nn.Conv2d(b,b,3,padding=1),nn.GroupNorm(4,b),nn.ReLU())
    def forward(self,x):return self.net(x)
class RoadUNet(nn.Module):
    def __init__(self):
        super().__init__();self.e1=Block(6,16);self.e2=Block(16,32);self.e3=Block(32,64);self.mid=Block(64,128)
        self.d3=Block(192,64);self.d2=Block(96,32);self.d1=Block(48,16);self.head=nn.Conv2d(16,3,1)
    def forward(self,x):
        a=self.e1(x);b=self.e2(nn.functional.max_pool2d(a,2));c=self.e3(nn.functional.max_pool2d(b,2))
        z=self.mid(nn.functional.max_pool2d(c,2))
        for skip,block in [(c,self.d3),(b,self.d2),(a,self.d1)]:
            z=nn.functional.interpolate(z,size=skip.shape[-2:],mode="bilinear",align_corners=False)
            z=block(torch.cat([z,skip],dim=1))
        return self.head(z)
model=RoadUNet().to(DEVICE)
freq=counts["train"]/counts["train"].sum()
weights=np.clip(1/np.sqrt(freq),1,30);weights=weights/weights.mean()
WEIGHTS=torch.tensor(weights,dtype=torch.float32,device=DEVICE)
def loss_fn(logits,y):
    valid=y!=IGNORE
    if not valid.any():return logits.sum()*0
    ce=nn.functional.cross_entropy(logits,y,weight=WEIGHTS,ignore_index=IGNORE)
    p=logits.softmax(1); terms=[]
    for c in [1,2]:
        target=(y==c).float(); pc=p[:,c]*valid
        terms.append(1-(2*(pc*target).sum()+1)/(pc.sum()+target.sum()+1))
    return ce+.5*torch.stack(terms).mean()
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}")


## 7 · Train, checkpoint, and resume
The best checkpoint is selected using **validation road mean IoU** (intersection-over-union of classes 1 and 2). The independent test split is used only after model selection. Training stops after six epochs without improvement. Checkpoints preserve optimizer and random-number states.

Rerun after a disconnected session from cell 1; cached downloads are reused if the runtime survives. With a new runtime, the next-day shortcut reloads your saved data and checkpoint from Drive. Files already downloaded are reused. Only load checkpoints created by this notebook from your own trusted folder.


In [ ]:
def confusion_update(cm,pred,y):
    valid=y!=IGNORE
    return cm+torch.bincount(3*y[valid]+pred[valid],minlength=9).reshape(3,3).cpu().numpy()
def metrics(cm):
    tp=np.diag(cm).astype(float);actual=cm.sum(1);predicted=cm.sum(0);union=actual+predicted-tp
    div=lambda a,b:np.divide(a,b,out=np.full_like(a,np.nan,dtype=float),where=b>0)
    return pd.DataFrame({"class":CLASS_NAMES,"IoU":div(tp,union),"precision":div(tp,predicted),
                         "recall":div(tp,actual),"Dice_F1":div(2*tp,actual+predicted),"support_pixels":actual})
@torch.no_grad()
def evaluate(loader):
    model.eval();cm=np.zeros((3,3),np.int64);total=0.;n=0
    for x,y in loader:
        x=x.to(DEVICE);y=y.to(DEVICE);logits=model(x)
        total+=loss_fn(logits,y).item();n+=1
        cm=confusion_update(cm,logits.argmax(1),y)
    return total/max(n,1),cm
config={"seed":SEED,"tile_size":TILE_SIZE,"patch_size":PATCH_SIZE,"road_buffer_metres":ROAD_BUFFER_METRES,
        "profile":PROFILE,"batch_size":BATCH_SIZE,"model":"six_channel_unet_v1","split_keys":manifest[["pre_key","post_key","split"]].to_dict("records")}
signature=hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()
(OUT/"config.json").write_text(json.dumps(config,indent=2))
optimizer=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)
history=[];start=0;best=-1.;stale=0
last=OUT/"last_checkpoint.pt";best_path=OUT/"best_model.pt"
if RESUME and last.exists():
    ck=torch.load(last,map_location="cpu",weights_only=False)
    if ck["signature"]!=signature:raise ValueError("Checkpoint settings differ. Choose a new RUN_NAME.")
    model.load_state_dict(ck["model"]);optimizer.load_state_dict(ck["optimizer"])
    for state in optimizer.state.values():
        for k,v in state.items():
            if torch.is_tensor(v):state[k]=v.to(DEVICE)
    start=ck["epoch"]+1;best=ck["best"];stale=ck["stale"];history=ck["history"]
    random.setstate(ck["random"]);np.random.set_state(ck["numpy"]);torch.set_rng_state(ck["torch"])
    if DEVICE.type=="cuda" and ck.get("cuda") is not None:torch.cuda.set_rng_state_all(ck["cuda"])
    print("Resuming at epoch",start+1)

def atomic_save(obj,path):
    tmp=path.with_suffix(".tmp");torch.save(obj,tmp);tmp.replace(path)
for epoch in range(start,EPOCHS):
    if stale>=6:print("Early stopping reached.");break
    began=time.time();model.train();losses=[]
    for x,y in loaders["train"]:
        x=x.to(DEVICE);y=y.to(DEVICE);optimizer.zero_grad(set_to_none=True)
        loss=loss_fn(model(x),y)
        if not torch.isfinite(loss):raise RuntimeError("Non-finite training loss; inspect the data.")
        loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1.0);optimizer.step();losses.append(loss.item())
    val_loss,cm=evaluate(loaders["val"]);table=metrics(cm)
    score=float(table.iloc[1:]["IoU"].mean())
    row={"epoch":epoch+1,"train_loss":float(np.mean(losses)),"val_loss":val_loss,"val_road_mIoU":score,"seconds":time.time()-began}
    history.append(row)
    if score>best:
        best=score;stale=0;atomic_save({"model":model.state_dict(),"signature":signature,"epoch":epoch+1,"config":config},best_path)
    else:stale+=1
    atomic_save({"model":model.state_dict(),"optimizer":optimizer.state_dict(),"epoch":epoch,"best":best,"stale":stale,
                 "history":history,"signature":signature,"random":random.getstate(),"numpy":np.random.get_state(),
                 "torch":torch.get_rng_state(),"cuda":torch.cuda.get_rng_state_all() if DEVICE.type=="cuda" else None},last)
    pd.DataFrame(history).to_csv(OUT/"history.csv",index=False)
    print(f"Epoch {epoch+1:02d}/{EPOCHS}: loss {row['train_loss']:.4f} | val road mIoU {score:.3f} | {row['seconds']:.0f}s")
if not best_path.exists():raise RuntimeError("No checkpoint saved. Check training and validation classes.")
model.load_state_dict(torch.load(best_path,map_location=DEVICE,weights_only=False)["model"])
fig,ax=plt.subplots(1,2,figsize=(12,4));hist=pd.DataFrame(history)
hist.plot(x="epoch",y=["train_loss","val_loss"],ax=ax[0]);hist.plot(x="epoch",y="val_road_mIoU",ax=ax[1])
plt.tight_layout();plt.savefig(OUT/"training_curves.png",dpi=160);plt.show()


## 8 · Independent evaluation and simple baseline
Report **flooded-road recall/precision/IoU**, not pixel accuracy dominated by background. The all-background baseline demonstrates why accuracy alone is misleading. Undefined metrics are displayed as missing, never converted into perfect scores.

The confidence diagnostic bins maximum softmax scores over the union of predicted/labelled road pixels. It is a calibration diagnostic, not a guarantee of safety. Test results must not be used to keep tuning this run; further tuning requires a new untouched test set.


In [ ]:
test_loss,test_cm=evaluate(loaders["test"])
results=metrics(test_cm);display(results)
results.to_csv(OUT/"test_metrics.csv",index=False)
base_cm=np.zeros((3,3),np.int64);base_cm[:,0]=test_cm.sum(1)
baseline=metrics(base_cm);baseline.to_csv(OUT/"all_background_baseline.csv",index=False)
print("Flooded-road IoU — trained:",results.iloc[2].IoU,"| all-background:",baseline.iloc[2].IoU)
fig,ax=plt.subplots(figsize=(7,5));im=ax.imshow(test_cm,cmap="Blues")
ax.set(xticks=range(3),yticks=range(3),xticklabels=["BG","Road","Flooded"],yticklabels=["BG","Road","Flooded"],xlabel="Predicted",ylabel="Label",title="Independent Louisiana test · pixel confusion matrix")
for i in range(3):
    for j in range(3):ax.text(j,i,f"{test_cm[i,j]:,}",ha="center",color="white" if test_cm[i,j]>test_cm.max()/2 else "black")
fig.colorbar(im);plt.tight_layout();plt.savefig(OUT/"confusion_matrix.png",dpi=160);plt.show()
bins=np.zeros((10,3),float)
model.eval()
with torch.no_grad():
    for x,y in loaders["test"]:
        p=model(x.to(DEVICE)).softmax(1).cpu();conf,pred=p.max(1)
        keep=(y!=IGNORE)&((y>0)|(pred>0))
        scores=conf[keep].numpy();correct=(pred[keep]==y[keep]).numpy()
        ids=np.minimum((scores*10).astype(int),9)
        for b in range(10):
            sel=ids==b;bins[b]+=[sel.sum(),scores[sel].sum(),correct[sel].sum()]
cal=pd.DataFrame({"bin_lower":np.arange(10)/10,"count":bins[:,0],"mean_score":np.divide(bins[:,1],bins[:,0],out=np.full(10,np.nan),where=bins[:,0]>0),
                  "observed_accuracy":np.divide(bins[:,2],bins[:,0],out=np.full(10,np.nan),where=bins[:,0]>0)})
cal.to_csv(OUT/"road_confidence_diagnostic.csv",index=False);display(cal)


## 9 · Prediction, image overlay, and GeoTIFF export
Choose any **test** tile. The output shows predicted road classes and the raw flooded-road model score. These are not calibrated road-closure probabilities.

Exports preserve the reference CRS and transform at the resized resolution. Nodata is 255 in class rasters and -9999 in score rasters. A high score does not establish current flooding; all dataset observations are historical.


In [ ]:
@torch.no_grad()
def infer_pair(x):
    model.eval()
    return model(torch.from_numpy(x.copy()).float().unsqueeze(0).to(DEVICE)/255.).softmax(1)[0].cpu().numpy()

def predict_tile(index=0,save=True):
    row=manifest.query('split=="test"').iloc[index]
    with np.load(row.cache) as z:x=z["x"];y=z["y"];trans=Affine(*z["transform"]);crs=str(z["crs"])
    valid=y!=IGNORE;p=infer_pair(x);pred=p.argmax(0).astype("uint8");pred[~valid]=IGNORE
    fig,ax=plt.subplots(1,4,figsize=(19,5))
    ax[0].imshow(x[:3].transpose(1,2,0));ax[0].set_title("Before")
    ax[1].imshow(x[3:].transpose(1,2,0));ax[1].set_title("After")
    ax[2].imshow(x[3:].transpose(1,2,0));overlay=np.zeros((*pred.shape,4));road=(pred==1)|(pred==2)
    overlay[road,:3]=PALETTE[pred[road]]/255.;overlay[road,3]=.75
    ax[2].imshow(overlay);ax[2].set_title("Prediction: teal road / coral flooded")
    heat=ax[3].imshow(np.ma.masked_where(~valid,p[2]),vmin=0,vmax=1,cmap="magma");ax[3].set_title("Flooded-road model score")
    fig.colorbar(heat,ax=ax[3],fraction=.04)
    for a in ax:a.axis("off")
    fig.suptitle(f"RAASTA · historical test tile {row.tile} · research prediction");plt.tight_layout()
    stem=row.area+"_"+row.tile
    if save:
        fig.savefig(OUT/(stem+"_prediction.png"),dpi=180,bbox_inches="tight")
        common={"driver":"GTiff","height":TILE_SIZE,"width":TILE_SIZE,"count":1,"crs":crs,"transform":trans,"compress":"deflate"}
        with rasterio.open(OUT/(stem+"_classes.tif"),"w",**common,dtype="uint8",nodata=IGNORE) as f:f.write(pred,1)
        score=p[2].copy();score[~valid]=-9999
        with rasterio.open(OUT/(stem+"_flood_score.tif"),"w",**common,dtype="float32",nodata=-9999) as f:f.write(score,1)
    plt.show()
    return row,p,pred,trans,crs
selected_row,selected_scores,selected_pred,selected_transform,selected_crs=predict_tile(0)


In [ ]:
import ipywidgets as widgets
picker=widgets.Dropdown(options=[(r.tile,i) for i,r in enumerate(manifest.query('split=="test"').itertuples())],description="Test tile:")
button=widgets.Button(description="Predict & export",button_style="info");panel=widgets.Output()
def on_predict(_):
    global selected_row,selected_scores,selected_pred,selected_transform,selected_crs
    with panel:
        panel.clear_output(wait=True)
        selected_row,selected_scores,selected_pred,selected_transform,selected_crs=predict_tile(picker.value)
button.on_click(on_predict)
display(widgets.HBox([picker,button]),panel)


## 10 · Geographic map of the selected prediction
Run this cell again after changing the selected tile. The raster overlay is warped to Web Mercator for correct placement on the web map. The underlying basemap is contextual and may be from a different date. Colours show **model predictions on historical imagery**, not present road status. Map tiles require internet access.


In [ ]:
import folium
from rasterio.warp import calculate_default_transform
from rasterio.transform import array_bounds
h,w=selected_pred.shape
bounds=array_bounds(h,w,selected_transform)
dst_transform,dw,dh=calculate_default_transform(selected_crs,"EPSG:3857",w,h,*bounds)
projected=np.full((dh,dw),IGNORE,np.uint8)
reproject(selected_pred,projected,src_transform=selected_transform,src_crs=selected_crs,
          dst_transform=dst_transform,dst_crs="EPSG:3857",resampling=Resampling.nearest,src_nodata=IGNORE,dst_nodata=IGNORE)
west,south,east,north=transform_bounds("EPSG:3857","EPSG:4326",*array_bounds(dh,dw,dst_transform))
rgba=np.zeros((dh,dw,4),np.uint8)
for c in [1,2]:rgba[projected==c]=[*PALETTE[c],185]
m=folium.Map(location=[(south+north)/2,(west+east)/2],zoom_start=15)
folium.raster_layers.ImageOverlay(rgba,bounds=[[south,west],[north,east]],name="Historical predictions · not live",opacity=.8).add_to(m)
folium.LayerControl().add_to(m)
folium.Marker([(south+north)/2,(west+east)/2],tooltip="Research only: historical SpaceNet imagery",popup="Teal: road without predicted flood. Coral: predicted flooded road. Neither is a driving-safety assessment.").add_to(m)
m.save(str(OUT/"historical_prediction_map.html"));display(m)


## 11 · Optional: predict a user-supplied image pair
Set `RUN_CUSTOM_UPLOAD=True` and run this cell to upload two georeferenced, 8-bit RGB GeoTIFFs. Specify which is before/after. Both must cover the same place; the post image is reprojected to the pre grid. Ordinary phone photos and screenshots are not compatible inputs. No local accuracy can be claimed without independent labels.


In [ ]:
RUN_CUSTOM_UPLOAD = False
if RUN_CUSTOM_UPLOAD:
    from google.colab import files
    uploaded=files.upload()
    names=list(uploaded)
    if len(names)!=2:raise ValueError("Upload exactly two georeferenced RGB GeoTIFF files.")
    pre_choice=widgets.Dropdown(options=names,description="Before:")
    post_choice=widgets.Dropdown(options=names,value=names[1],description="After:")
    run_custom=widgets.Button(description="Predict pair");custom_panel=widgets.Output()
    def custom_predict(_):
        with custom_panel:
            custom_panel.clear_output(wait=True)
            if pre_choice.value==post_choice.value:raise ValueError("Choose different before/after files.")
            x,valid,t,c=load_pair(pre_choice.value,post_choice.value);p=infer_pair(x)
            fig,axes=plt.subplots(1,3,figsize=(14,4))
            axes[0].imshow(x[:3].transpose(1,2,0));axes[0].set_title("Before")
            axes[1].imshow(x[3:].transpose(1,2,0));axes[1].set_title("After")
            axes[2].imshow(np.ma.masked_where(~valid,p[2]),vmin=0,vmax=1,cmap="magma");axes[2].set_title("Uncalibrated flooded-road score")
            for a in axes:a.axis("off")
            plt.tight_layout();fig.savefig(OUT/"custom_pair_prediction.png",dpi=160);plt.show()
    run_custom.on_click(custom_predict);display(pre_choice,post_choice,run_custom,custom_panel)
else: print("Optional custom upload disabled. Test-set predictions above need no upload.")


## 12 · Inspection-planning experiment — SYNTHETIC NETWORK
This demonstrates how visual predictions could support an operational decision. It is **not** an extracted real road network. Nodes, demands, inspection costs, and the assignment of image scores to fictional road links are synthetic. No real community counts or delivery savings are claimed.

For the selected tile, mean flooded-road scores are sampled along annotated road corridors (geometry only; flood labels are not used here). These uncalibrated scores are assigned to edges of a small fictional graph as **illustrative closure probabilities**. If too few annotated roads exist, the example pauses rather than inventing model outputs.

For each candidate inspection, simulate possible states of other edges and measure how much verifying that edge resolves uncertainty about reachable destination demand. Divide that by an assumed inspection cost. This is a simple value-of-information heuristic under independent edge states, not a validated logistics optimizer. Correlated flooding, inspection safety, vehicle limits, uncertain demand, and real coordinates are not modelled.


In [ ]:
import networkx as nx

def source_road_scores(row,p):
    geoms=road_geometries(local_path(row.label_key),selected_crs)
    scores=[]
    for geometry,_unused_label in geoms:
        mask=rasterize([(geometry,1)],out_shape=selected_pred.shape,transform=selected_transform,dtype="uint8")>0
        mask &= selected_pred!=IGNORE
        if mask.sum()>=4:scores.append(float(p[2][mask].mean()))
    return scores

scores=source_road_scores(selected_row,selected_scores)
G=nx.Graph()
positions={"Depot":(0,1),"J1":(1,2),"J2":(1,0),"A":(2,2),"B":(2,0),"Clinic":(3,1)}
edges=[("Depot","J1"),("Depot","J2"),("J1","A"),("J2","B"),("J1","J2"),("A","Clinic"),("B","Clinic")]
demand={"A":40,"B":60,"Clinic":80} # fictional demand units; not people
rng=np.random.default_rng(SEED)
if len(scores)<len(edges):
    print("Select a different test tile with at least seven annotated road features; rerun cells 9/10/12.")
    planning=None
else:
    for i,(u,v) in enumerate(edges):G.add_edge(u,v,p=float(np.clip(scores[i],.05,.95)),cost=1+i%3)
    def reachable(state):
        H=nx.Graph();H.add_nodes_from(G.nodes)
        H.add_edges_from([e for e,closed in zip(edges,state) if not closed])
        connected=nx.node_connected_component(H,"Depot")
        return sum(d for n,d in demand.items() if n in connected)
    probabilities=np.array([G[u][v]["p"] for u,v in edges])
    states=rng.random((400,len(edges)))<probabilities
    rows=[]
    for i,(u,v) in enumerate(edges):
        differences=[]
        for state in states:
            opened=state.copy();closed=state.copy();opened[i]=False;closed[i]=True
            differences.append(reachable(opened)-reachable(closed))
        delta=float(np.mean(differences));p=probabilities[i];cost=G[u][v]["cost"]
        rows.append({"fictional_link":f"{u}—{v}","illustrative_closure_probability":p,
                     "conditional_access_change":delta,"assumed_inspection_cost":cost,
                     "verification_priority":p*(1-p)*delta/cost})
    planning=pd.DataFrame(rows).sort_values("verification_priority",ascending=False)
    display(planning);planning.to_csv(OUT/"SYNTHETIC_inspection_ranking.csv",index=False)
    fig,ax=plt.subplots(figsize=(9,5));nx.draw(G,positions,with_labels=True,node_color="#8dd3c7",node_size=1800,ax=ax)
    nx.draw_networkx_edge_labels(G,positions,edge_labels={(u,v):f"score {G[u][v]['p']:.2f}" for u,v in edges},ax=ax)
    ax.set_title("SYNTHETIC access experiment · fictional locations and link states")
    plt.tight_layout();plt.savefig(OUT/"SYNTHETIC_access_network.png",dpi=160);plt.show()
    print("Suggested first verification in this simulation:",planning.iloc[0].fictional_link)


## 13 · Export results and prepare an honest demonstration
The archive includes your best trained model, metrics, charts, manifest, configuration, generated maps, and synthetic-experiment outputs. The resumable optimizer checkpoint is kept in the output folder but excluded from the smaller shareable archive. Raw imagery is not bundled.

For your teacher: explain the six input channels, three mask classes, geographic split, U-Net, class imbalance, IoU, and the difference between visual flooding and road passability. Show a successful and a failed prediction.

For LinkedIn: report the measured flooded-road IoU/precision/recall and dataset region. Label the logistics graph as a simulation. Do not claim Pakistan deployment, live satellite coverage, guaranteed safe routes, or saved lives.


In [ ]:
card={"project":"Raasta","author":"Allah Ditta","status":"research prototype; locally executed results",
      "dataset":"SpaceNet 8 historical Germany/Louisiana","training_region":"Germany",
      "test_region":"Louisiana-East","pakistan_validation":"not performed",
      "model":"six-channel U-Net; trained from scratch","classes":CLASS_NAMES,
      "road_label_buffer_radius_metres":ROAD_BUFFER_METRES,"best_validation_road_mIoU":best,
      "test_metrics":json.loads(results.to_json(orient="records")),
      "limitations":["Non-flood-labelled roads are not verified passable", "Road widths are a fixed label proxy",
                      "Domain shift untested in Pakistan", "Scores not calibrated closure probabilities",
                      "Access-planning graph, demands and costs are synthetic"],
      "sources":["https://www.spacenet.ai/sn8-challenge/","https://github.com/SpaceNetChallenge/SpaceNet8"]}
(OUT/"model_card.json").write_text(json.dumps(card,indent=2,allow_nan=False))
import zipfile
archive=WORK/(RUN_NAME+"_results.zip")
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    for p in OUT.iterdir():
        if p.is_file() and p.name!="last_checkpoint.pt" and p.suffix!=".tmp":z.write(p,p.name)
print("Saved results:",OUT);print("Downloadable archive:",archive)
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:pass


## Troubleshooting and next research steps
- **403/404 dataset response:** rerun discovery. If persistent, check the official SpaceNet page; do not substitute unrelated data or pairing files.
- **GPU unavailable / memory error:** use batch size 2 or 1 and start a new run name. The smoke profile is only a functionality check.
- **No flooded pixels:** increase the tile caps. Very small geographic subsets may omit classes.
- **Poor Louisiana performance:** inspect failures and improve training on a separate development set. Do not repeatedly tune against the held-out test split.
- **Interrupted session:** reconnect, run install/configuration and the next-day shortcut, then jump to training or prediction. Keep the original run settings.
- **Map unavailable:** prediction plots and GeoTIFF exports work without online basemap tiles.

### What is implemented versus future work
Implemented: actual official-file discovery/download, image registration, mask creation, train/validation/test split, U-Net training, checkpoints, measured evaluation, prediction exports, map, and a synthetic inspection-priority experiment.

Future work: a real connected road graph, dated Pakistani image/field labels, calibration on a local development set, field-report ingestion, inspection-strategy benchmarking, user validation, and any operational deployment. The synthetic graph is deliberately separated from the geographic prediction map.

### References and attribution
- [SpaceNet 8 challenge and dataset](https://www.spacenet.ai/sn8-challenge/) — dataset listed as CC BY-SA 4.0. Preserve attribution when sharing imagery or derived dataset artifacts.
- [Official SpaceNet 8 baseline and winning solutions](https://github.com/SpaceNetChallenge/SpaceNet8).
- Hänsch et al., *SpaceNet 8 — The Detection of Flooded Roads and Buildings*, CVPR Workshops, 2022. [Paper](https://openaccess.thecvf.com/content/CVPR2022W/EarthVision/papers/Hansch_SpaceNet_8_-_The_Detection_of_Flooded_Roads_and_Buildings_CVPRW_2022_paper.pdf).
- [Logistics Cluster — LogIE](https://logcluster.org/en/what-we-do/emergency-response/logie) — operational context.

This notebook's buffer width, compact model, sampling policy and toy decision heuristic are educational design choices, not an official benchmark reproduction.


In [ ]:
# RAASTA interface — run after training, or after restoring your previous session.
# This cell adds a UI without changing any earlier cells or the training model.
import copy as _rui_copy
import html as _rui_html
import uuid as _rui_uuid
import zipfile as _rui_zipfile
from pathlib import Path as _rui_Path
import ipywidgets as _rui_widgets
from IPython.display import display as _rui_display, HTML as _rui_HTML, clear_output as _rui_clear

def _rui_build():
    needed = ['model', 'DEVICE', 'OUT', 'load_pair', 'np', 'torch', 'plt', 'rasterio', 'Affine']
    missing = [name for name in needed if name not in globals()]
    if missing:
        print('Run installation and configuration, then training OR the Continue a previous session cell first.')
        print('Missing setup: ' + ', '.join(missing))
        return
    try:
        from google.colab import output as colab_output
        colab_output.enable_custom_widget_manager()
    except ImportError:
        pass
    test_rows = []
    if 'manifest' in globals():
        test_rows = list(manifest[manifest['split'] == 'test'].itertuples())
    mode = _rui_widgets.ToggleButtons(options=['Saved test example', 'Upload image pair'], description='Input:')
    examples = _rui_widgets.Dropdown(
        options=[(str(r.area) + ' / ' + str(r.tile), i) for i, r in enumerate(test_rows)],
        description='Example:', layout=_rui_widgets.Layout(width='95%'))
    pre = _rui_widgets.FileUpload(accept='.tif,.tiff', multiple=False, description='Before image')
    post = _rui_widgets.FileUpload(accept='.tif,.tiff', multiple=False, description='After image')
    uploads = _rui_widgets.VBox([_rui_widgets.HTML(
        'Upload overlapping, georeferenced <b>8-bit RGB GeoTIFFs</b> of the same place: before and after flooding. '
        'Maximum 100 MiB per file. Ordinary photos and screenshots are not supported.'), pre, post])
    run = _rui_widgets.Button(description='Run prediction', icon='play', button_style='success')
    download = _rui_widgets.Button(description='Download results', icon='download', disabled=True)
    status = _rui_widgets.HTML('Ready. Choose an input and click Run prediction.')
    result = _rui_widgets.Output()
    state = {'archive': None}

    def switch(change=None):
        custom = mode.value == 'Upload image pair'
        examples.layout.display = 'none' if custom else ''
        uploads.layout.display = '' if custom else 'none'
    mode.observe(switch, names='value')
    switch()

    def uploaded_file(widget, destination):
        value = widget.value
        if not value:
            raise ValueError('Select both a before image and an after image.')
        # ipywidgets 7 uses a dictionary; ipywidgets 8 uses a tuple of records.
        item = next(iter(value.values())) if isinstance(value, dict) else value[0]
        name = item.get('name', item.get('metadata', {}).get('name', 'image.tif'))
        if _rui_Path(name).suffix.lower() not in {'.tif', '.tiff'}:
            raise ValueError('Use .tif or .tiff imagery.')
        content = item['content']
        if not 0 < len(content) <= 100 * 1024 * 1024:
            raise ValueError('Each image must be nonempty and at most 100 MiB.')
        destination.write_bytes(bytes(content))
        return str(name)

    def predict(button):
        run.disabled = True
        download.disabled = True
        state['archive'] = None
        status.value = 'Loading the saved best model and preparing images…'
        with result:
            _rui_clear(wait=True)
            try:
                checkpoint = _rui_Path(OUT) / 'best_model.pt'
                if not checkpoint.exists():
                    raise ValueError('No best_model.pt found. Finish at least one successful training epoch first.')
                ck = torch.load(checkpoint, map_location='cpu', weights_only=True)
                if 'signature' in globals() and ck.get('signature') != signature:
                    raise ValueError('Saved model belongs to a different configuration. Restore the correct run.')
                # Use a separate model so training state and earlier predictions remain untouched.
                ui_model = _rui_copy.deepcopy(model).to(DEVICE)
                ui_model.load_state_dict(ck['model'])
                ui_model.eval()
                folder = _rui_Path(OUT) / 'ui_predictions' / _rui_uuid.uuid4().hex[:12]
                folder.mkdir(parents=True, exist_ok=False)
                truth = None
                if mode.value == 'Upload image pair':
                    before_name = uploaded_file(pre, folder / 'before.tif')
                    after_name = uploaded_file(post, folder / 'after.tif')
                    x, valid, transform, crs = load_pair(folder / 'before.tif', folder / 'after.tif')
                    source = {'kind': 'user upload', 'before': before_name, 'after': after_name}
                else:
                    if examples.value is None:
                        raise ValueError('No saved test examples available. Restore the dataset or upload a pair.')
                    row = test_rows[examples.value]
                    with np.load(row.cache) as z:
                        x = z['x'].copy()
                        truth = z['y'].copy()
                        valid = truth != 255
                        transform = Affine(*z['transform'])
                        crs = str(z['crs'])
                    source = {'kind': 'historical test example', 'area': str(row.area), 'tile': str(row.tile)}
                if not valid.any():
                    raise ValueError('No valid overlapping pixels found.')
                with torch.inference_mode():
                    tensor = torch.from_numpy(np.ascontiguousarray(x)).float().unsqueeze(0).to(DEVICE) / 255.
                    scores = ui_model(tensor).softmax(1)[0].cpu().numpy()
                del ui_model, tensor
                if scores.shape != (3, *valid.shape) or not np.isfinite(scores).all():
                    raise ValueError('Model produced invalid output. Check the saved model and imagery.')
                classes = scores.argmax(0).astype('uint8')
                classes[~valid] = 255
                confidence = float(scores.max(0)[valid].mean() * 100)
                accuracy = float((classes[valid] == truth[valid]).mean() * 100) if truth is not None else None
                accuracy_text = f'{accuracy:.2f}%' if accuracy is not None else 'Unavailable — uploaded images have no reference labels'
                _rui_display(_rui_HTML('<p><b>Selected image pixel accuracy:</b> ' + accuracy_text + '<br><b>Mean prediction confidence:</b> ' + f'{confidence:.2f}%' + ' (uncalibrated; not accuracy)</p>'))
                road = (classes == 1) | (classes == 2)
                flooded = classes == 2
                share = float(flooded.sum() / road.sum() * 100) if road.any() else None
                summary = dict(source=source, checkpoint=str(checkpoint), checkpoint_epoch=ck.get('epoch'),
                    pixel_accuracy_percent=accuracy, mean_prediction_confidence_percent=confidence,
                    valid_pixels=int(valid.sum()), predicted_road_pixels=int(road.sum()),
                    predicted_flooded_road_pixels=int(flooded.sum()),
                    flooded_share_of_predicted_road_pixels_percent=share,
                    note='Research prediction; pixel shares are not road length or verified road safety. Scores are uncalibrated. Pakistan performance has not been validated.')
                import json as ui_json
                (folder / 'summary.json').write_text(ui_json.dumps(summary, indent=2))
                share_text = f'{share:.1f}%' if share is not None else 'N/A — no road pixels predicted'
                _rui_display(_rui_HTML('<h3>Prediction results</h3><p><b>Flooded share of predicted road pixels:</b> '
                    + share_text + '<br><b>Valid image coverage:</b> ' + f'{valid.mean()*100:.1f}%'
                    + '<br>Teal: road with no flood label · Coral: flooded road</p>'))
                fig, axes = plt.subplots(1, 4, figsize=(19, 5))
                axes[0].imshow(x[:3].transpose(1, 2, 0)); axes[0].set_title('Before')
                axes[1].imshow(x[3:].transpose(1, 2, 0)); axes[1].set_title('After')
                overlay = np.zeros((*valid.shape, 4))
                overlay[classes == 1] = [41/255, 182/255, 164/255, .75]
                overlay[flooded] = [250/255, 92/255, 78/255, .8]
                axes[2].imshow(x[3:].transpose(1, 2, 0)); axes[2].imshow(overlay)
                axes[2].set_title('Predicted road classes')
                heat = axes[3].imshow(np.ma.masked_where(~valid, scores[2]), cmap='magma', vmin=0, vmax=1)
                axes[3].set_title('Flooded-road model score')
                fig.colorbar(heat, ax=axes[3], fraction=.046)
                for ax in axes: ax.axis('off')
                fig.tight_layout()
                fig.savefig(folder / 'prediction.png', dpi=160, bbox_inches='tight')
                plt.show(); plt.close(fig)
                common = dict(driver='GTiff', height=valid.shape[0], width=valid.shape[1], count=1,
                              crs=crs, transform=transform, compress='deflate')
                with rasterio.open(folder / 'classes.tif', 'w', **common, dtype='uint8', nodata=255) as dst:
                    dst.write(classes, 1)
                flood_score = scores[2].astype('float32').copy(); flood_score[~valid] = -9999
                with rasterio.open(folder / 'flood_score.tif', 'w', **common, dtype='float32', nodata=-9999) as dst:
                    dst.write(flood_score, 1)
                archive = folder / 'raasta_results.zip'
                with _rui_zipfile.ZipFile(archive, 'w', _rui_zipfile.ZIP_DEFLATED) as z:
                    for name in ['prediction.png', 'classes.tif', 'flood_score.tif', 'summary.json']:
                        z.write(folder / name, arcname=name)
                state['archive'] = archive
                download.disabled = False
                status.value = '<b>Complete.</b> Results saved to ' + _rui_html.escape(str(folder))
            except Exception as exc:
                status.value = '<b>Could not predict:</b> ' + _rui_html.escape(str(exc))
            finally:
                run.disabled = False

    def save_results(button):
        if state['archive'] is not None:
            try:
                from google.colab import files as ui_files
                ui_files.download(str(state['archive']))
            except Exception as exc:
                status.value = 'Results are saved at ' + _rui_html.escape(str(state['archive'])) + '. Download error: ' + _rui_html.escape(str(exc))
    run.on_click(predict)
    download.on_click(save_results)
    _rui_display(_rui_widgets.VBox([
        _rui_widgets.HTML('<h2 style="color:#159b8d">RAASTA · Flooded-road prediction</h2>'
            '<p>Choose an example or upload a before-and-after satellite image pair.</p>'
            '<p><small>Research prototype using historical SpaceNet 8 data. Predictions do not establish road safety. '
            'Performance in Pakistan has not been validated.</small></p>'),
        mode, examples, uploads, _rui_widgets.HBox([run, download]), status, result],
        layout=_rui_widgets.Layout(border='1px solid #cbd5e1', padding='20px', width='100%')))

_rui_build()


In [ ]:
# Model accuracy on the held-out test set (run after training or session restore).
# Background can inflate pixel accuracy: also report road/flood IoU and recall.
import ipywidgets as _acc_widgets
from IPython.display import display as _acc_display, clear_output as _acc_clear
_acc_button = _acc_widgets.Button(description='Evaluate model accuracy', button_style='info', layout=_acc_widgets.Layout(width='230px'))
_acc_output = _acc_widgets.Output()
def _acc_run(button):
    button.disabled = True
    with _acc_output:
        _acc_clear(wait=True)
        try:
            import copy as _acc_copy
            import json as _acc_json
            ck = torch.load(OUT/'best_model.pt', map_location='cpu', weights_only=True)
            if ck.get('signature') != signature:
                raise ValueError('Checkpoint configuration mismatch. Restore the correct session.')
            net = _acc_copy.deepcopy(model).to(DEVICE).eval()
            net.load_state_dict(ck['model'])
            cm = np.zeros((3,3), dtype=np.int64)
            with torch.inference_mode():
                for x, y in loaders['test']:
                    prediction = net(x.to(DEVICE)).argmax(1).cpu()
                    cm = confusion_update(cm, prediction, y.cpu())
            del net
            if not cm.sum(): raise ValueError('No valid test pixels.')
            table = metrics(cm)
            accuracy = float(np.trace(cm)/cm.sum()*100)
            balanced = float(table['recall'].mean()*100)
            report = {'test_pixel_accuracy_percent': accuracy,
                      'test_balanced_accuracy_percent': balanced,
                      'background_only_accuracy_percent': float(cm[0].sum()/cm.sum()*100),
                      'test_road_mean_IoU_percent': float(table.iloc[1:]['IoU'].mean()*100),
                      'checkpoint_epoch': ck.get('epoch')}
            print(f"Held-out test pixel accuracy: {accuracy:.2f}%")
            print(f"Balanced accuracy (mean class recall): {balanced:.2f}%")
            print(f"Background-only baseline accuracy: {report['background_only_accuracy_percent']:.2f}%")
            print('Per-class metrics below are fractions from 0 to 1. Accuracy is measured against dataset labels, not a guarantee for new locations.')
            _acc_display(table)
            (OUT/'accuracy_summary.json').write_text(_acc_json.dumps(report, indent=2))
            table.to_csv(OUT/'accuracy_per_class.csv', index=False)
            print('Accuracy results saved in your run folder.')
        except Exception as exc:
            print('Evaluation unavailable:', str(exc))
            print('Run training or restore the previous session before evaluating.')
        finally:
            button.disabled = False
_acc_button.on_click(_acc_run)
_acc_display(_acc_button, _acc_output)
